## Introduction to instruction fine-tuning

### Preparing a dataset for supervised instruction fine-tuning

In [71]:
import json 
import os 
import urllib

def download_and_load_file(file_path, url): 
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response: 
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file: 
            file.write(text_data)
    else:	#1
        with open(file_path, "r", encoding="utf-8") as file: 
            text_data = file.read()
    with open(file_path, "r") as file: 
        data = json.load(file)
    return data

file_path = "instruction-data.json" 
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch" 
       "/main/ch07/01_main-chapter-code/instruction-data.json"
)


data = download_and_load_file(file_path, url) 
print("Number of entries:", len(data))


Number of entries: 1100


In [4]:
print("Example entry:\n", data[50])

Example entry:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


In [5]:
print("Example entry:\n", data[999])

Example entry:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


In [72]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task."
        f"Write a response that approapriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    return instruction_text+input_text

In [73]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input+desired_response)

Below is an instruction that describes a task.Write a response that approapriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


In [74]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input+desired_response)

Below is an instruction that describes a task.Write a response that approapriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
The correct spelling is 'Occasion.'


In [75]:
train_portion = int(len(data)*0.85)
test_portion = int(len(data)*0.1)
val_portion = len(data)-train_portion-test_portion

train_data = data[:train_portion]
test_data = data[train_portion:train_portion+test_portion]
val_data = data[train_portion+test_portion:]

print(len(train_data))
print(len(test_data))
print(len(val_data))

935
110
55


### Organizing data into training batches

In [76]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self,data,tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\\n### Response:\\n{entry['output']}"
            full_text = instruction_plus_input +response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self,index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [10]:
# Padding will be done according to longest one in each batch, not entire training data

In [77]:
def custome_collate_draft_1(batch, pad_token_id=50256,device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst = []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id] 

        padded = (
            new_item + [pad_token_id]*(batch_max_length-len(new_item))
        )

        inputs = torch.tensor(padded[:-1])
        inputs_lst.append(inputs)
    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor

In [78]:
inputs_1 = [0,1,2,3,4]
inputs_2 = [5,6]
inputs_3 = [7,8,9]
batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
print(custome_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


In [79]:
def custome_collate_draft_2(batch, pad_token_id=50256,device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, target_lst = [],[]

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id] 

        padded = (
            new_item + [pad_token_id]*(batch_max_length-len(new_item))
        )

        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        inputs_lst.append(inputs)
        target_lst.append(targets)
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(target_lst).to(device)
    return inputs_tensor,targets_tensor
    

In [80]:
inputs,targets = custome_collate_draft_2(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


In [81]:
# excpet the first instance of padded token remaining will be made -100 to remove them from training

In [82]:
def custome_collate_fn(batch, pad_token_id=50256,ignore_index=-100,allowed_max_length=None,device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, target_lst = [],[]

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id] 

        padded = (
            new_item + [pad_token_id]*(batch_max_length-len(new_item))
        )

        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = targets==pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel()>1:
            targets[indices[1:]]=ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]
        
        inputs_lst.append(inputs)
        target_lst.append(targets)
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(target_lst).to(device)
    return inputs_tensor,targets_tensor
    

In [83]:
inputs,targets = custome_collate_fn(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


In [84]:
logits_1 = torch.tensor(
    [[-1.0,1.0],
    [-0.5,1.5]]
)
targets_1 = torch.tensor([0,1])
loss_1 = torch.nn.functional.cross_entropy(logits_1,targets_1)
loss_1

tensor(1.1269)

In [85]:
logits_1 = torch.tensor(
    [[-1.0,1.0],
    [-0.5,1.5],
    [-0.5,1.5]]
)
targets_1 = torch.tensor([0,1,1])
loss_1 = torch.nn.functional.cross_entropy(logits_1,targets_1)
loss_1

tensor(0.7936)

In [86]:
logits_1 = torch.tensor(
    [[-1.0,1.0],
    [-0.5,1.5],
    [-0.5,1.5]]
)
targets_1 = torch.tensor([0,1,-100])
loss_1 = torch.nn.functional.cross_entropy(logits_1,targets_1)
loss_1

tensor(1.1269)

### Creating data loaders for an instruction dataset

In [87]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [88]:
from functools import partial

customized_collate_fn = partial(
    custome_collate_fn,
    device=device,
    allowed_max_length=1024
)

In [89]:
import tiktoken
tokenzier = tiktoken.get_encoding("gpt2")

In [90]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenzier)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

val_dataset = InstructionDataset(val_data, tokenzier)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenzier)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [35]:
print("Train Loader:")
for inputs,target in train_loader:
    print(inputs.shape, target.shape)


Train Loader:
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 81]) torch.Size([8, 81])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 74]) torch.Size([8, 74])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 86]) torch.Size([8, 86])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 81]) torch.Size([8, 81])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 74]) torch.Size([8, 74])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 82]) torch.Size([8, 82])
torch.Size([8, 74]) torch.Size([8, 74])
torch.Size([8, 84]) torch.Size([8, 84])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 88]) torch.Size([8, 88])
torch.Size([8, 74]) torch.Size([8, 74])
torch.Size([8, 85]) torch.Size([8, 85])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 73]) torch.

### Loading a pretrained LLM

In [44]:
from gpt_download import download_and_load_gpt2
from Chapter4 import GPTModel
from Chapter5 import load_weights_into_gpt
torch.cuda.empty_cache()
BASE_CONFIG = {
"vocab_size": 50257,	# Vocabulary size 
"context_length": 1024, # Context length 
    "drop_rate": 0.0,	# oropout rate 
    "qkv_bias": True	# Query-key-value bias
}

model_configs = {
"gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
"gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
"gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
"gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MOoEL = "gpt2-small (124M)" 
BASE_CONFIG.update(model_configs[CHOOSE_MOoEL])
model_size = CHOOSE_MOoEL.split(" ")[-1].lstrip("(").rstrip(")") 
settings, params = download_and_load_gpt2(
model_size=model_size,
models_dir="gpt2"
)

model = GPTModel(BASE_CONFIG) 
load_weights_into_gpt(model, params) 
model.eval();


File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe


In [45]:
torch.manual_seed(123)
input_text = format_input(val_data[0])
print(input_text)

Below is an instruction that describes a task.Write a response that approapriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'


In [58]:
from Chapter5n import generate, text_to_token_ids, token_ids_to_text

token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text,tokenzier),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256,

)
generated_text = token_ids_to_text(token_ids,tokenzier)

RuntimeError: Boolean value of Tensor with more than one value is ambiguous

In [59]:
response_text = generated_text[len(input_text):].strip()
print(response_text)

### Instruction:

Convert the active sentence to passive: 'The chef cooks the meal every day.'

### Instruction:

Convert the active


### Fine tuning the LLM on instruction data

In [60]:
from Chapter5n import calc_loss_loader, train_model_simple

In [61]:
model.to(device)
torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(
        train_loader, model,device,num_batches=5
    )
    val_loss = calc_loss_loader(
        val_loader,model,device,num_batches=5
    )

print("training loss:", train_loss)
print("Validation loss:", val_loss)

training loss: 4.963017082214355
Validation loss: 4.835745334625244


In [68]:
import time

start_time = time.time()
torch.manual_seed(123)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=0.00005, weight_decay=0.1
)

num_epochs=2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer,device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenzier
)

end_time = time.time()
execution_time_minutes = (end_time-start_time)/60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
